# 🤖 Notebook 3 — Full LangGraph Agent
### Tools + Memory + Real Use Case — Travel & Utility Assistant

**What you'll learn:**
- Combine Tools + Memory into a full agent
- Understand the complete agent loop
- Add a System Prompt for agent persona
- Visualise the graph
- Run a real multi-turn demo
- Debug message traces

---
> **Prerequisite:** Notebooks 1 (Tools) and 2 (Memory) complete!
---

## 📦 Setup

In [ ]:
!pip install -qU langgraph langchain langchain-openai requests

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your-api-key-here"  # 🔁 Replace

---
## 🧩 The Complete Agent Architecture

```
User Message
     │
     ▼
┌────────────┐    tool_calls?=YES    ┌───────────┐
│ agent_node │ ──────────────────►  │ ToolNode  │
│   (LLM)   │ ◄──────────────────  │ (execute) │
└─────┬──────┘    tool result       └───────────┘
      │
  tool_calls?=NO
      │
      ▼
     END  →  Final Answer to User
      │
      ▼
 MemorySaver  →  Checkpoint saved for thread_id
```

| Component | Role |
|-----------|------|
| `agent_node` | LLM reads messages + decides what to do |
| `ToolNode` | Executes the tool chosen by LLM |
| `should_continue` | Routes: tool call → tools, else → END |
| `MemorySaver` | Saves full message history per `thread_id` |
| `SystemMessage` | Gives the agent a persona and instructions |

---
## 🛠️ Step 1 — Define All Tools

In [ ]:
from langchain_core.tools import tool
import requests, math

@tool
def get_weather(location: str) -> str:
    """Get current weather for any city.
    Args: location: City name e.g. 'Mumbai', 'Delhi'
    """
    url      = f"https://wttr.in/{location}?format=j1"
    data     = requests.get(url, timeout=10).json()
    current  = data["current_condition"][0]
    return (
        f"Weather in {location}: {current['weatherDesc'][0]['value']}, "
        f"Temp: {current['temp_C']}°C, Feels like: {current['FeelsLikeC']}°C, "
        f"Humidity: {current['humidity']}%"
    )

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression. Args: expression e.g. '2+2', 'sqrt(144)'"""
    try:
        allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
        return f"{expression} = {eval(expression, {'__builtins__': {}}, allowed)}"
    except Exception as e:
        return f"Error: {e}"

@tool
def get_trip_plan(origin: str, destination: str) -> str:
    """Suggest travel options between two cities.
    Args: origin: departure city, destination: arrival city
    """
    plans = {
        ("mumbai", "goa"):   "🚆 Konkan Railway ~8hrs | ✈️ Flight ~1hr | 🚗 NH66 ~10hrs. 💡 Book train 60 days ahead!",
        ("mumbai", "delhi"): "✈️ Flight ~2hrs (best) | 🚆 Rajdhani ~16hrs overnight. 💡 Great food on Rajdhani!",
        ("delhi",  "agra"):  "🚄 Gatimaan Express ~1.5hrs | 🚗 Yamuna Expressway ~3hrs. 💡 Taj Mahal at sunrise!",
    }
    key  = (origin.lower(), destination.lower())
    plan = plans.get(key) or plans.get((destination.lower(), origin.lower()))
    return f"{origin} → {destination}: {plan}" if plan else f"Check IRCTC / Skyscanner for {origin} → {destination}"

@tool
def currency_converter(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert currency. Args: amount, from_currency e.g.'USD', to_currency e.g.'INR'"""
    rates = {"USD": 83.5, "EUR": 90.2, "GBP": 105.8, "AED": 22.7, "SGD": 62.1, "INR": 1.0}
    f, t  = from_currency.upper(), to_currency.upper()
    if f not in rates or t not in rates:
        return f"Unsupported. Use: {list(rates.keys())}"
    return f"{amount} {f} = {amount * rates[f] / rates[t]:.2f} {t} (approx)"

all_tools = [get_weather, calculate, get_trip_plan, currency_converter]
print("✅ 4 tools ready!")

---
## 🧠 Step 2 — State, LLM, System Prompt

In [ ]:
from typing import Annotated, Sequence, TypedDict
import operator
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

# State — messages list that APPENDS (not replaces)
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# LLM bound to tools
llm            = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
llm_with_tools = llm.bind_tools(all_tools)

# System prompt — agent persona
SYSTEM_PROMPT = SystemMessage(content="""You are a helpful travel and utility assistant for India.
You help with: weather, travel planning, currency conversion, and math calculations.
You have memory of the full conversation — use previous context when user refers to it.
Be friendly, concise, and helpful. Always use tools for real-time data.
""")

print("✅ State, LLM and System Prompt configured!")

---
## 🏗️ Step 3 — Build the Graph

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# ── agent_node: LLM decides what to do ──────────
def agent_node(state: AgentState):
    messages = [SYSTEM_PROMPT] + list(state["messages"])
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# ── should_continue: routing logic ──────────────
def should_continue(state: AgentState):
    last = state["messages"][-1]
    if last.tool_calls:
        tool_name = last.tool_calls[0]["name"]
        print(f"   🔧 Tool called: {tool_name}({last.tool_calls[0]['args']})")
        return "tools"
    return END

# ── Build graph ──────────────────────────────────
def create_agent():
    builder = StateGraph(AgentState)

    builder.add_node("agent", agent_node)           # Node 1: LLM
    builder.add_node("tools", ToolNode(all_tools))  # Node 2: Tool executor

    builder.add_edge(START, "agent")                # Entry: always start at agent
    builder.add_conditional_edges(                  # After agent: tools or END
        "agent",
        should_continue,
        ["tools", END]
    )
    builder.add_edge("tools", "agent")              # After tools: back to agent

    return builder.compile(checkpointer=MemorySaver())

app = create_agent()
print("✅ Full agent graph compiled!")

---
## 👁️ Step 4 — Visualise the Graph

In [ ]:
from IPython.display import Image, display
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print(app.get_graph().draw_mermaid())

---
## 💬 Step 5 — Chat Function

In [ ]:
def chat(user_input: str, thread_id: str = "default") -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config
    )
    return result["messages"][-1].content

print("✅ Chat function ready!")

---
## 🚀 Step 6 — Individual Tool Demos

In [ ]:
# ── Weather ──────────────────────────────────────
print("🌤️  Weather Demo")
print("-" * 45)
q = "What's the weather in Mumbai?"
print(f"👤 {q}")
print(f"🤖 {chat(q, 'demo_weather')}")

In [ ]:
# ── Calculator ───────────────────────────────────
print("🧮 Calculator Demo")
print("-" * 45)
q = "What is sqrt(225) + 15 * 4?"
print(f"👤 {q}")
print(f"🤖 {chat(q, 'demo_calc')}")

In [ ]:
# ── Trip Planner ─────────────────────────────────
print("🗺️  Trip Planner Demo")
print("-" * 45)
q = "How can I travel from Mumbai to Goa?"
print(f"👤 {q}")
print(f"🤖 {chat(q, 'demo_trip')}")

In [ ]:
# ── Currency ─────────────────────────────────────
print("💱 Currency Demo")
print("-" * 45)
q = "Convert 1000 USD to INR"
print(f"👤 {q}")
print(f"🤖 {chat(q, 'demo_currency')}")

---
## 🧠 Step 7 — Full Memory + Tools Demo

Watch how the agent uses **both memory and tools** across multiple turns.

In [ ]:
SESSION = "full_demo"

turns = [
    # Turn 1 — Tool: weather
    "What's the weather in Mumbai right now?",

    # Turn 2 — Tool: trip
    "I want to travel from Mumbai to Goa. What are my options?",

    # Turn 3 — Memory: remembers cities from Turn 1 & 2
    "Which cities have we discussed so far?",

    # Turn 4 — Tool: currency
    "I have 800 USD for the trip. How much is that in INR?",

    # Turn 5 — Tool: calculator + memory (uses INR from Turn 4)
    "If my hotel in Goa costs 3500 INR per night, how many nights can I afford?",

    # Turn 6 — Tool: weather for destination
    "What's the weather in Goa? Is it a good time to visit?",

    # Turn 7 — Full memory test
    "Give me a summary of everything we planned in this conversation.",
]

for i, question in enumerate(turns, 1):
    print(f"\n{'═'*58}")
    print(f" Turn {i}: {question}")
    print(f"{'═'*58}")
    answer = chat(question, thread_id=SESSION)
    print(f"🤖 {answer}")

---
## 🔍 Step 8 — Debug: Full Message Trace

In [ ]:
# See ALL messages for one query — what happens inside the graph
config = {"configurable": {"thread_id": "debug_trace"}}

result = app.invoke(
    {"messages": [HumanMessage(content="What is the weather in Delhi and convert 200 USD to INR?")]},
    config=config
)

print("📋 Full message trace:\n")
for i, msg in enumerate(result["messages"]):
    role = type(msg).__name__
    print(f"[{i}] {role}")
    if msg.content:
        print(f"    content    : {msg.content[:120]}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"    tool_calls : {msg.tool_calls}")
    if hasattr(msg, "name") and msg.name:
        print(f"    tool name  : {msg.name}")
    print()

---
## 🔄 Step 9 — Memory Isolation Test

In [ ]:
# Session A — Rahul's conversation
chat("My name is Rahul and I'm travelling to Goa.", thread_id="rahul_session")
ans_a = chat("What is my name and where am I going?", thread_id="rahul_session")

# Session B — Fresh session, no memory of A
ans_b = chat("What is my name and where am I going?", thread_id="new_session")

print("🔵 Rahul's session:")
print(ans_a)    # Should know name and destination ✅
print()
print("🟢 New session:")
print(ans_b)    # Should say it doesn't know ✅

---
## 📊 Step 10 — Inspect Stored State

In [ ]:
config = {"configurable": {"thread_id": "full_demo"}}
state  = app.get_state(config)

messages = state.values["messages"]
print(f"Total messages stored in 'full_demo' session: {len(messages)}")
print()
for i, msg in enumerate(messages):
    role    = type(msg).__name__
    content = msg.content[:60] if msg.content else "[tool call]"
    print(f"[{i:2d}] {role:<15} → {content}")

---
## 🎓 Final Summary

```
Full Agent = Tools + Memory + Graph

Tools     → Python functions LLM can call (@tool decorator)
Memory    → MemorySaver + thread_id persists message history
Graph     → StateGraph wires agent_node ↔ ToolNode in a loop
```

**Agent Loop:**
```
User → agent_node → tool_calls? → YES → ToolNode → agent_node (loop)
                                → NO  → END (final answer)
```

**Key components:**

| Code | Purpose |
|------|---------|
| `@tool` | Defines a tool |
| `llm.bind_tools()` | Registers tools with LLM |
| `ToolNode(all_tools)` | Executes tool calls |
| `MemorySaver()` | Enables cross-turn memory |
| `thread_id` | Isolates conversations |
| `operator.add` | Appends messages (not replace) |
| `SystemMessage` | Agent persona / instructions |

---
## 🏋️ Final Exercises

1. **Add a new tool** — e.g. `get_flight_price(origin, destination)` with dummy data
2. **Multi-tool in one query** — ask something that needs weather + calculator together
3. **Change persona** — modify SYSTEM_PROMPT to make agent respond in Hindi
4. **Count tool calls** — how many tool calls happened in the Turn 7 summary? Inspect the trace
5. **Build your own use case** — create a customer support agent with 3 custom tools